# Data Loading & AggBar Exploration

This notebook is a **deep dive into data handling** in Factorium:

1. Downloading data with `BinanceDataLoader`
2. Understanding the `AggBar` container
3. Time-bar aggregation at different intervals
4. Slicing, filtering, and exporting data
5. Working with different data formats (Polars / Pandas / CSV / Parquet)

**Prerequisites**: `pip install factorium`

## 1. Downloading Data with BinanceDataLoader

`BinanceDataLoader` fetches historical trade data from [Binance Vision](https://data.binance.vision/) and aggregates it into OHLCV bars on the fly.

In [ ]:
from factorium import BinanceDataLoader, AggBar

import matplotlib.pyplot as plt
import pandas as pd
import polars as pl

%matplotlib inline
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["figure.dpi"] = 100

In [ ]:
loader = BinanceDataLoader()

# Load 7 days of 1-minute time bars for 5 symbols
agg = loader.load_aggbar(
    symbols=["BTCUSDT", "ETHUSDT", "BNBUSDT", "SOLUSDT", "XRPUSDT"],
    data_type="aggTrades",
    market_type="futures",
    futures_type="um",
    days=7,
    bar_type="time",       # Time-based bars
    interval=60_000,       # 1 minute = 60,000 ms
)

print(f"Type: {type(agg).__name__}")
print(f"Total bars: {len(agg):,}")
print(f"Symbols: {agg.symbols}")
print(f"Columns: {agg.cols}")

## 2. Understanding AggBar

`AggBar` is a **multi-symbol OHLCV data container** that stores data in **long format**:

| Column | Description |
|--------|-------------|
| `start_time` | Bar open timestamp (epoch ms) |
| `end_time` | Bar close timestamp (epoch ms) |
| `symbol` | Trading pair identifier |
| `open` | Opening price |
| `high` | Highest price |
| `low` | Lowest price |
| `close` | Closing price |
| `volume` | Trading volume |

In [ ]:
# View as Polars DataFrame
agg.to_polars().head(10)

In [ ]:
# View as Pandas DataFrame
agg.to_df().head(10)

In [ ]:
# Per-symbol summary: bar count, time range, missing data
agg.info()

In [ ]:
# Metadata
meta = agg.metadata
print(f"Number of rows:  {meta.num_rows:,}")
print(f"Number of symbols: {len(meta.symbols)}")
print(f"Time range: {meta.min_time} → {meta.max_time}")

## 3. Extracting Factors from AggBar

Use `agg["column_name"]` to extract a column as a `Factor` object. This is the starting point for all computations.

In [ ]:
close = agg["close"]
volume = agg["volume"]

print(f"close factor: {type(close).__name__}, name='{close.name}', rows={len(close):,}")
print(f"volume factor: {type(volume).__name__}, name='{volume.name}', rows={len(volume):,}")

# Preview factor data
close.data.head()

In [ ]:
# Extract multiple columns as a new AggBar
ohlc = agg[["open", "high", "low", "close"]]
print(f"Sub-AggBar columns: {ohlc.cols}")
print(f"Sub-AggBar bars: {len(ohlc):,}")

## 4. Slicing Data

`AggBar.slice()` lets you filter by **time range** and/or **symbols** to create a new AggBar subset.

In [ ]:
# Slice by symbols only
btc_only = agg.slice(symbols=["BTCUSDT"])
print(f"BTC only: {len(btc_only):,} bars, symbols={btc_only.symbols}")

# Slice by multiple symbols
btc_eth = agg.slice(symbols=["BTCUSDT", "ETHUSDT"])
print(f"BTC+ETH: {len(btc_eth):,} bars, symbols={btc_eth.symbols}")

## 5. Different Time Intervals

By changing the `interval` parameter, you can aggregate raw trade data into bars of different durations.

In [ ]:
# 1-minute bars (already loaded)
print(f"1-min bars: {len(agg):,}")

# 5-minute bars
agg_5m = loader.load_aggbar(
    symbols=["BTCUSDT", "ETHUSDT"],
    data_type="aggTrades",
    market_type="futures",
    futures_type="um",
    days=7,
    bar_type="time",
    interval=300_000,   # 5 minutes = 300,000 ms
)
print(f"5-min bars: {len(agg_5m):,}")

# 1-hour bars
agg_1h = loader.load_aggbar(
    symbols=["BTCUSDT", "ETHUSDT"],
    data_type="aggTrades",
    market_type="futures",
    futures_type="um",
    days=7,
    bar_type="time",
    interval=3_600_000,  # 1 hour = 3,600,000 ms
)
print(f"1-hour bars: {len(agg_1h):,}")

In [ ]:
# Compare bar counts
comparison = pd.DataFrame({
    "Interval": ["1 min", "5 min", "1 hour"],
    "interval_ms": [60_000, 300_000, 3_600_000],
    "Total Bars": [len(agg.slice(symbols=["BTCUSDT", "ETHUSDT"])), len(agg_5m), len(agg_1h)],
})
comparison

In [ ]:
# Visualize the same price data at different resolutions
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=False)

for ax, (label, data) in zip(axes, [
    ("1-min", agg.slice(symbols=["BTCUSDT"])),
    ("5-min", agg_5m.slice(symbols=["BTCUSDT"])),
    ("1-hour", agg_1h.slice(symbols=["BTCUSDT"])),
]):
    df = data.to_df()
    df["datetime"] = pd.to_datetime(df["start_time"], unit="ms")
    ax.plot(df["datetime"], df["close"], linewidth=0.8)
    ax.set_title(f"BTCUSDT Close Price — {label} bars ({len(data):,} bars)")
    ax.set_ylabel("Price")
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Date")
plt.tight_layout()
plt.show()

## 6. Saving and Loading Data

`AggBar` can be exported to CSV or Parquet and loaded back.

In [ ]:
from pathlib import Path

# Save to Parquet (efficient binary format)
output_dir = Path("./sample_data")
output_dir.mkdir(exist_ok=True)

agg.to_parquet(output_dir / "crypto_1min.parquet")
print(f"Saved to {output_dir / 'crypto_1min.parquet'}")

# Save to CSV
agg.to_csv(output_dir / "crypto_1min.csv")
print(f"Saved to {output_dir / 'crypto_1min.csv'}")

In [ ]:
# Load back from CSV
agg_loaded = AggBar.from_csv(output_dir / "crypto_1min.csv")
print(f"Loaded from CSV: {len(agg_loaded):,} bars, symbols={agg_loaded.symbols}")

In [ ]:
# You can also create AggBar from a Polars or Pandas DataFrame
df = agg.to_polars()
agg_from_df = AggBar.from_df(df)
print(f"From DataFrame: {len(agg_from_df):,} bars")

## 7. Using ResearchSession for Quick Analysis

`ResearchSession` wraps an AggBar and provides a high-level API. It can also be created directly from files.

In [ ]:
from factorium import ResearchSession

# Create from AggBar
session = ResearchSession(agg, default_frequency="1m")

print(f"Session symbols: {session.symbols}")
print(f"Session columns: {session.cols}")

# Quick factor creation and analysis
signal = session.factor("close").cs_rank()
report = session.quick_report(signal)
print(report)

In [ ]:
# ResearchSession can also load from files directly
session_from_csv = ResearchSession.from_csv(
    output_dir / "crypto_1min.csv",
    default_frequency="1m",
)
print(f"Session from CSV: {len(session_from_csv.symbols)} symbols")

## 8. Cleanup

Remove sample data files created during this notebook.

In [ ]:
import shutil

if output_dir.exists():
    shutil.rmtree(output_dir)
    print(f"Cleaned up {output_dir}")

## Summary

In this notebook we covered:
- **`BinanceDataLoader`**: Download historical trade data and aggregate into time bars
- **`AggBar`**: Multi-symbol OHLCV container with `info()`, `slice()`, `to_polars()`, `to_df()`
- **Time intervals**: Aggregating the same data into 1-min, 5-min, and 1-hour bars
- **Persistence**: Saving to CSV/Parquet and loading back
- **`ResearchSession`**: High-level API that wraps AggBar for streamlined research

**Next notebooks:**
- `01_momentum_factor_research.ipynb` — Full factor research workflow
- `02_mean_reversion_factor.ipynb` — Mean reversion with volatility normalization
- `04_multi_factor_combination.ipynb` — Combine multiple factors